In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Memory experiment analysis: min-distance + ROC curves.
Refactored into a single organized script. 
Modify only `run_experiment_at_noise` to explore new dynamics.
"""

# ===================== Imports =====================
import sys, os, glob, json, math, datetime
from collections import defaultdict

import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

# project-specific paths
sys.path.append('/orcd/data/jhm/001/om2/jmhicks/projects/TextureStreaming/code/')
sys.path.append('../utls/')
sys.path.append('../src/model/')
sys.path.append("/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/")

from chexture_choolbox.auditorytexture.texture_model import TextureModel
from chexture_choolbox.auditorytexture.helpers import FlattenStats
from texture_prior.params import model_params, statistics_set, texture_dataset
from texture_prior.utils import path

import DistanceMemoryModel
import encoders
from utls.plotting import ensure_dir
from utls.loading import load_results_with_exclusion_2, move_sequences_to_used


# ===================== Config =====================
NV_DEFAULT   = 0.2      # per-dim noise std per drift step
SEED         = 123
DEVICE       = 'cuda' if torch.cuda.is_available() else 'cpu'
PC_DIMS      = 256
TIMES_TO_PLOT = [10, 17, 22, 40, 80, 119]

import matplotlib.pyplot as plt

def make_model_save_dir(base_dir, model_info):
    """
    Create and return a directory path based on model information.
    """
    date_tag = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    parts = [
        model_info.get('model_name', 'unknown'),
        model_info.get('metric', 'metric'),
        model_info.get('noise_mode', 'noise'),
        model_info.get('encoder', 'encoder'),
        f"rate{model_info.get('rate', 'NA')}",
        model_info.get('run_id', date_tag)
    ]
    subdir = "_".join(str(p) for p in parts if p)
    save_dir = os.path.join(base_dir, subdir)
    os.makedirs(save_dir, exist_ok=True)

    # Save metadata if not already there
    info_path = os.path.join(save_dir, "info.json")
    if not os.path.exists(info_path):
        with open(info_path, "w") as f:
            json.dump(model_info, f, indent=2)

    return save_dir


def save_all_figures(figs, save_dir, prefix=""):
    """
    Save a list of Matplotlib figures to the given directory.

    Args:
        figs      : list of matplotlib.figure.Figure
        save_dir  : str, target directory
        prefix    : str, filename prefix
    """
    for i, fig in enumerate(figs, start=1):
        name = f"{prefix}Fig{i:02d}"
        figpath_png = os.path.join(save_dir, f"{name}.png")
        figpath_pdf = os.path.join(save_dir, f"{name}.pdf")
        fig.savefig(figpath_png, dpi=300, bbox_inches="tight")
        fig.savefig(figpath_pdf, bbox_inches="tight")
        print(f"Saved {figpath_png} and {figpath_pdf}")


def save_single_figure(fig, save_dir, filename_base):
    """
    Save a single Matplotlib figure as PNG and PDF.
    """
    png_path = os.path.join(save_dir, f"{filename_base}.png")
    pdf_path = os.path.join(save_dir, f"{filename_base}.pdf")
    fig.savefig(png_path, dpi=300, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")
    print(f"Saved {png_path} and {pdf_path}")

# ===================== Utilities =====================
def roc_from_arrays(hit_scores, fa_scores, score_type="distance"):
    """
    Compute ROC curve and AUC given hit and FA scores.

    Args:
        hit_scores (np.ndarray): scores for hits
        fa_scores  (np.ndarray): scores for false alarms
        score_type (str): 
            "distance"   -> smaller = more similar (flip sign for ROC)
            "likelihood" -> larger = more similar (use directly)

    Returns:
        fpr, tpr, auc_val
    """
    if hit_scores.size == 0 or fa_scores.size == 0:
        return None

    y_true = np.concatenate([
        np.ones_like(hit_scores), 
        np.zeros_like(fa_scores)
    ])
    y_score = np.concatenate([hit_scores, fa_scores])

    if score_type == "distance":
        # lower = better → flip sign so higher = better
        y_score = -y_score
    elif score_type == "likelihood":
        # higher = better → leave as is
        pass
    else:
        raise ValueError(f"Unknown score_type: {score_type}")

    fpr, tpr, _ = roc_curve(y_true, y_score)
    return fpr, tpr, auc(fpr, tpr)

def plot_roc(ax, fpr, tpr, label):
    ax.plot(fpr, tpr, label=label)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)


# ===================== Flexible Runners =====================

# def run_experiment_at_noise(
#     NV_value,
#     *,
#     X0, name_to_idx, experiment_list,
#     DEVICE="cpu", DistanceMemoryModel=None, zscore_projector=None
# ):
#     """Default runner using DistanceMemoryModel (min distance)."""
#     hit_dists, fa_dists = [], []
#     isi_hit_dists = defaultdict(list)
#     T_max = max((len(seq) for seq in experiment_list), default=0)
#     fa_by_t = [[] for _ in range(T_max)]

#     for seq in experiment_list:
#         if not seq:
#             continue

#         model = DistanceMemoryModel.DistanceMemoryModel(
#             encoding_model=zscore_projector,
#             noise_variance=float(NV_value),
#             criterion=0.0,
#             device=DEVICE
#         )
#         model.memory_bank = []
#         seq_idx = [name_to_idx[fn] for fn in seq]
#         bank_set, last_seen = set(), {}

#         with torch.no_grad():
#             for t, idx_incoming in enumerate(seq_idx, start=1):
#                 if len(model.memory_bank) > 0:
#                     model.apply_noise_to_memory()

#                 if len(model.memory_bank) > 0:
#                     bank_mat = torch.stack([rep.view(-1) for rep in model.memory_bank], dim=0)
#                     probe = X0[idx_incoming].view(1, -1)
#                     dmin = torch.linalg.norm(bank_mat - probe, dim=1).min().item()

#                     if idx_incoming in bank_set:
#                         hit_dists.append(dmin)
#                         isi = t - last_seen[idx_incoming]
#                         isi_hit_dists[isi].append((dmin, t))
#                     else:
#                         fa_dists.append(dmin)
#                         fa_by_t[t-1].append(dmin)

#                 model.memory_bank.append(X0[idx_incoming].detach().clone())
#                 bank_set.add(idx_incoming)
#                 last_seen[idx_incoming] = t

#     return {
#         "hits": np.asarray(hit_dists, float),
#         "fas": np.asarray(fa_dists, float),
#         "isi_hit_dists": isi_hit_dists,
#         "fa_by_t": fa_by_t,
#         "T_max": T_max,
#         "score_type": "distance"
#     }



def run_experiment_scores(
    sigma0, *, rate=None, metric="mahalanobis", noise_mode="diffuse",
    X0=None, name_to_idx=None, experiment_list=None, debug=False, **kwargs
):
    assert metric in {"mahalanobis", "loglikelihood", "euclidean"}
    assert noise_mode in {"diffuse", "decay", "power-decay", "exp-decay"}

    hit_scores, fa_scores = [], []
    isi_hit_dists = defaultdict(list)
    T_max = max((len(seq) for seq in experiment_list), default=0)
    fa_by_t = [[] for _ in range(T_max)]
    D = X0.shape[1]
    dim_std = X0.std(0, unbiased=True)
    max_std = dim_std.max()  # global maximum across all features
    scaled_std = dim_std / max_std  # normalize so max = 1.0

    # 🔹 store all std values across updates
    stds_over_time = []  # list of (t, std)

    for seq in experiment_list:
        if not seq:
            continue

        seq_idx = [name_to_idx[fn] for fn in seq]
        memory_bank, bank_set, last_seen = [], set(), {}

        for t, idx_incoming in enumerate(seq_idx, start=1):
            probe = X0[idx_incoming].view(1, -1)
            scores = []

            # --- UPDATE EACH MEMORY (drift step) ---
            for mem in memory_bank:
                age = t - mem["t_inserted"]
                if age <= 0:
                    continue

                # compute std according to schedule
                if noise_mode == "diffuse":
                    std = sigma0 * np.sqrt(age)
                elif noise_mode == "decay":
                    std = sigma0 / np.sqrt(age)
                elif noise_mode == "power-decay":
                    std = sigma0 * (age ** (rate if rate is not None else 0.5))
                elif noise_mode == "exp-decay":
                    std = sigma0 * np.exp(age ** (rate if rate is not None else 1.0))
                std = max(std, 1e-6)

                stds_over_time.append((t, std))  # 🔹 store std evolution

                # --- APPLY ADDITIVE NOISE TO MEMORY STATE ---
                noise = torch.randn_like(mem["mu"]) * (std * scaled_std)
                mem["mu"] = mem["mu"] + noise  # cumulative drift
                mem["age"] = age

                # --- SCORING ---
                diff = probe - mem["mu"]
                sqdist = torch.sum(diff**2).item()
                var = std**2

                if metric == "mahalanobis":
                    score = (sqdist**0.5) / std
                elif metric == "loglikelihood":
                    score = (
                        -0.5 * D * np.log(2 * np.pi)
                        - D * np.log(std)
                        - 0.5 * (sqdist / var)
                    )
                elif metric == "euclidean":
                    score = torch.norm(diff).item()

                scores.append(score)

            # --- DECISION + STORE NEW MEMORY ---
            if scores:
                score_val = max(scores) if metric == "loglikelihood" else min(scores)
                if idx_incoming in bank_set:
                    hit_scores.append(score_val)
                    isi = t - last_seen[idx_incoming]
                    isi_hit_dists[isi].append((score_val, t))
                else:
                    fa_scores.append(score_val)
                    fa_by_t[t-1].append(score_val)

            # store clean *plus initial noise* representation
            base_rep = X0[idx_incoming].detach().clone()
            initial_noise = torch.randn_like(base_rep) * (sigma0 * dim_std)
            noisy_mem = base_rep + initial_noise

            memory_bank.append({"mu": noisy_mem, "t_inserted": t})
            bank_set.add(idx_incoming)
            last_seen[idx_incoming] = t

    # convert to arrays for easy plotting later
    stds_over_time = np.array(stds_over_time, dtype=float)
    if stds_over_time.size > 0:
        ts, stds = stds_over_time[:, 0], stds_over_time[:, 1]
    else:
        ts, stds = np.array([]), np.array([])

    return {
        "hits": np.asarray(hit_scores, float),
        "fas": np.asarray(fa_scores, float),
        "isi_hit_dists": isi_hit_dists,
        "fa_by_t": fa_by_t,
        "T_max": T_max,
        "score_type": "likelihood" if metric == "loglikelihood" else "distance",
        "noise_mode": noise_mode,
        "stds_time": ts,
        "stds_values": stds,  # ✅ for plotting std(t)
    }

# ===================== Analysis Helpers =====================
def rocs_across_noise(
    noise_levels,
    *,
    runner,
    X0, name_to_idx, experiment_list,
    DistanceMemoryModel=None, zscore_projector=None, DEVICE="cpu", **kwargs
):
    curves, runs = {}, {}
    for nv in noise_levels:
        run = runner(
            nv,
            X0=X0, name_to_idx=name_to_idx, experiment_list=experiment_list,
            DistanceMemoryModel=DistanceMemoryModel, zscore_projector=zscore_projector,
            DEVICE=DEVICE, **kwargs
        )
        runs[nv] = run

        # 👇 use the runner’s declared score_type instead of guessing
        score_type = run.get("score_type", "distance")
        res = roc_from_arrays(run["hits"], run["fas"], score_type=score_type)

        if res is not None:
            curves[nv] = res
    return curves, runs


def roc_for_isi(run_data, isi_value):
    hits_raw = run_data["isi_hit_dists"].get(isi_value, [])
    if not hits_raw:
        return None

    hits = np.array([d for (d, t) in hits_raw], float)

    # Use all FAs (pooled across time) as comparison set
    fas = np.asarray(run_data["fas"], float)

    score_type = run_data.get("score_type", "distance")
    return roc_from_arrays(hits, fas, score_type=score_type)


def roc_for_second_half(run_data):
    T_half = run_data["T_max"] // 2
    hits, fas = [], []

    for lst in run_data["isi_hit_dists"].values():
        hits.extend([d for (d, t) in lst if t > T_half])
    hits = np.asarray(hits, float)

    for t_idx, bucket in enumerate(run_data["fa_by_t"], start=1):
        if t_idx > T_half:
            fas.extend(bucket)
    fas = np.asarray(fas, float)

    score_type = run_data.get("score_type", "distance")
    return roc_from_arrays(hits, fas, score_type=score_type)

# ===================== Plotting Wrappers =====================
def plot_noise_overlays(curves_by_noise, title="ROC curves across noise levels",
                        save_dir=None, prefix="NoiseOverlay"):
    """
    Plot and optionally save ROC overlays across noise levels.
    
    Args:
        curves_by_noise : dict {noise_level: (fpr, tpr, auc_val)}
        title           : str, title for the plot
        save_dir        : str or None, if provided, saves to this directory
        prefix           : str, filename prefix
    """
    fig, ax = plt.subplots(figsize=(6, 6))
    for nv, (fpr, tpr, auc_val) in curves_by_noise.items():
        plot_roc(ax, fpr, tpr, label=f"noise={nv:g} | AUC={auc_val:.3f}")
    ax.plot([0,1], [0,1], 'k--', alpha=0.5)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(title)
    ax.legend(loc="lower right")
    ax.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.show()

    # Save the overlay figure if requested
    if save_dir is not None:
        filename_base = f"{prefix}_ROC_overlay"
        save_single_figure(fig, save_dir, filename_base)


def plot_isi_and_half_for_noise(nv, run_data, isis=(1, 17)):
    fig, ax = plt.subplots(figsize=(6, 6))

    for k in isis:
        res = roc_for_isi(run_data, k)
        if res is not None:
            fpr, tpr, auc_val = res
            plot_roc(ax, fpr, tpr, label=f"ISI={k} | AUC={auc_val:.3f}")

    ax.plot([0,1], [0,1], 'k--', alpha=0.5)
    ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
    ax.set_title(f"Noise={nv:g}: ROC for each ISI")
    ax.legend(loc="lower right"); ax.grid(True, alpha=0.2)
    plt.tight_layout(); plt.show

from scipy.stats import norm

def auroc_to_dprime(auroc):
    """Convert AUROC to d′ via z-transform rule."""
    auroc = np.clip(auroc, 1e-6, 1 - 1e-6)
    return np.sqrt(2) * norm.ppf(auroc)

def compute_rates_by_isi_optimal(run_data):
    """
    Compute hit/FA rates by ISI using the ROC-based optimal criterion
    (threshold that minimizes distance to (0,1)).
    """
    isi_hit_dists = run_data["isi_hit_dists"]
    fa_by_t = run_data["fa_by_t"]
    score_type = run_data["score_type"]

    isis = sorted(isi_hit_dists.keys())
    hit_rates, fa_rates = [], []

    for isi in isis:
        hit_scores = np.asarray([s for (s, _) in isi_hit_dists[isi]], float)
        fa_scores  = np.asarray(fa_by_t[isi-1], float) if isi-1 < len(fa_by_t) else np.array([])

        if hit_scores.size == 0 or fa_scores.size == 0:
            hit_rates.append(np.nan)
            fa_rates.append(np.nan)
            continue

        # build ROC labels/scores
        y_true = np.concatenate([np.ones(len(hit_scores)), np.zeros(len(fa_scores))])
        scores = np.concatenate([hit_scores, fa_scores])

        # ROC expects higher score = "old"
        if score_type == "distance":
            scores = -scores

        fpr, tpr, thresholds = roc_curve(y_true, scores)

        # pick threshold closest to (0,1)
        dists = np.sqrt((fpr - 0)**2 + (tpr - 1)**2)
        i_best = np.argmin(dists)

        hit_rates.append(tpr[i_best])
        fa_rates.append(fpr[i_best])

    return isis, np.array(hit_rates), np.array(fa_rates)


# ---------- main plotting ----------

def plot_across_noise(noise_levels, runs, model_info=None, save_base=None, bins=60, xmax_percentile=99.5, isis=(1, 17)):
    """
    Make four multi-panel figures across noise levels:
      - Fig1: histograms of hits vs FAs (adaptive grid)
      - Fig2: ROC curves (adaptive grid)
      - Fig3: d′ vs ISI (using AUROC → d′ transform)
      - Fig4: Hit & FA rate vs ISI (optimal ROC criterion per ISI)
    """
    nlevels = len(noise_levels)

    # ===============================================
    # Figure 1: Histograms (adaptive layout)
    # ===============================================
    ncols = min(5, nlevels)
    nrows = math.ceil(nlevels / ncols)
    fig1, axes1 = plt.subplots(nrows, ncols, figsize=(6.5*ncols, 4.5*nrows))
    axes1 = np.atleast_2d(axes1)
    axes_flat = axes1.ravel()

    for ax, nv in zip(axes_flat, noise_levels):
        run_data = runs[nv]
        hits = run_data["hits"]
        fas  = run_data["fas"]
        score_type = run_data.get("score_type", "distance")

        pooled = np.concatenate([hits, fas]) if hits.size and fas.size else None
        if pooled is None or pooled.size == 0:
            ax.set_title(f"Noise={nv:g} (no data)")
            continue

        if score_type == "distance":
            xmax = float(np.percentile(pooled, xmax_percentile))
            rng = (0, xmax)
            xlabel = "Min distance"
        else:  # likelihood
            xmin = float(np.percentile(pooled, 100 - xmax_percentile))
            xmax = float(np.percentile(pooled, xmax_percentile))
            rng = (xmin, xmax)
            xlabel = "Log likelihood"

        ax.hist(hits, bins=bins, range=rng, density=True, alpha=0.55,
                color='green', label=f'hits (N={len(hits)})')
        ax.hist(fas,  bins=bins, range=rng, density=True, alpha=0.55,
                color='red',   label=f'FAs (N={len(fas)})')
        ax.set_title(f"Noise={nv:g}")
        ax.set_xlabel(xlabel); ax.set_ylabel("Density")
        ax.legend(fontsize=8)

    # Turn off any empty subplots
    for ax in axes_flat[len(noise_levels):]:
        ax.axis("off")

    fig1.suptitle("Hit vs FA score distributions across noise levels", fontsize=14)
    plt.tight_layout()
    plt.show()

    # ===============================================
    # Figure 2: ROC curves (adaptive layout)
    # ===============================================
    ncols = min(5, nlevels)
    nrows = math.ceil(nlevels / ncols)
    fig2, axes2 = plt.subplots(nrows, ncols, figsize=(6*ncols, 6*nrows))
    axes2 = np.atleast_2d(axes2)
    axes_flat = axes2.ravel()

    for ax, nv in zip(axes_flat, noise_levels):
        run_data = runs[nv]

        # ISI-specific ROCs
        for k in isis:
            res = roc_for_isi(run_data, k)
            if res is not None:
                fpr, tpr, auc_val = res
                plot_roc(ax, fpr, tpr, label=f"ISI={k} | AUC={auc_val:.3f}")

        ax.plot([0, 1], [0, 1], 'k--', alpha=0.5)
        ax.set_title(f"Noise={nv:g}")
        ax.set_xlabel("False Positive Rate")
        ax.set_ylabel("True Positive Rate")
        ax.legend(loc="lower right", fontsize=8)
        ax.grid(True, alpha=0.2)

    # Turn off unused ROC subplots
    for ax in axes_flat[len(noise_levels):]:
        ax.axis("off")

    fig2.suptitle("ISI ROC curves across noise levels", fontsize=14)
    plt.tight_layout()
    plt.show()
    
    def bootstrap_dprime_ci(run_data, isi_value, n_boot=200, ci=68):
        from sklearn.utils import resample
        """
        Compute bootstrap mean and SEM for d′ at a given ISI.
        
        Args:
            run_data : dict with "isi_hit_dists" and "fas"
            isi_value : ISI to analyze
            n_boot : number of bootstrap resamples
            ci : confidence interval percentage (default 68 ≈ 1 SEM)
        Returns:
            mean_dprime, sem_dprime
        """
        hits_raw = run_data["isi_hit_dists"].get(isi_value, [])
        if not hits_raw:
            return np.nan, np.nan
    
        hits = np.array([d for (d, t) in hits_raw], float)
        fas  = np.asarray(run_data["fas"], float)
        if len(hits) < 3 or len(fas) < 3:
            return np.nan, np.nan
    
        score_type = run_data.get("score_type", "distance")
        dprimes = []
    
        for _ in range(n_boot):
            h_bs = resample(hits, replace=True)
            f_bs = resample(fas, replace=True)
            res = roc_from_arrays(h_bs, f_bs, score_type=score_type)
            if res is not None:
                _, _, auc_val = res
                dprimes.append(auroc_to_dprime(auc_val))
    
        dprimes = np.array(dprimes)
        mean_d = np.nanmean(dprimes)
        sem_d  = np.nanstd(dprimes)
        return mean_d, sem_d
    
    # ===============================================
    # Figure 3: d′ vs ISI
    # ===============================================
    fig3, ax3 = plt.subplots(figsize=(7, 5))

    for nv in noise_levels:
        run_data = runs[nv]
        dprime_means, dprime_sems = [], []

        for k in isis:
            mean_d, sem_d = bootstrap_dprime_ci(run_data, k)
            dprime_means.append(mean_d)
            dprime_sems.append(sem_d)

        dprime_means = np.array(dprime_means)
        dprime_sems  = np.array(dprime_sems)

        # skip empty or nan data
        if np.all(np.isnan(dprime_means)):
            continue

        ax3.errorbar(
            isis, dprime_means, yerr=dprime_sems,
            marker='o', capsize=3, elinewidth=1, alpha=0.7,
            label=f"Noise={nv:g}"
        )

    ax3.set_title("d′ vs ISI across noise levels (with SEM)")
    ax3.set_xlabel("ISI")
    ax3.set_ylabel("d′ (z(AUROC))")
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # ===============================================
    # Figure 4: Hit & FA rates vs ISI
    # ===============================================
    fig4, axes4 = plt.subplots(1, 2, figsize=(12,5), sharex=True)
    ax_hit, ax_fa = axes4

    for nv in noise_levels:
        run_data = runs[nv]
        isis_avail, hit_rates, fa_rates = compute_rates_by_isi_optimal(run_data)
        ax_hit.plot(isis_avail, hit_rates, marker='o', label=f"Noise={nv:g}", alpha=0.7)
        ax_fa.plot(isis_avail, fa_rates, marker='s', label=f"Noise={nv:g}", alpha=0.7)

    ax_hit.set_title("Hit rate vs ISI (optimal ROC criterion)")
    ax_hit.set_xlabel("ISI"); ax_hit.set_ylabel("Hit rate")
    ax_hit.legend(); ax_hit.grid(True, alpha=0.3)

    ax_fa.set_title("False Alarm rate vs ISI (optimal ROC criterion)")
    ax_fa.set_xlabel("ISI"); ax_fa.set_ylabel("FA rate")
    ax_fa.legend(); ax_fa.grid(True, alpha=0.3)

    fig4.suptitle("Hit & False Alarm rates across ISI (optimal ROC criterion)", fontsize=14)
    plt.tight_layout()
    plt.show()


    # ===============================================
    # Figure 5: Theoretical STD schedule vs Time
    # ===============================================
    fig5, ax5 = plt.subplots(figsize=(7, 5))
    T_max = max(run_data["T_max"] for run_data in runs.values()) if runs else 120
    t = np.arange(1, T_max + 1)

    # Infer noise_mode and rate (if available)
    noise_mode = model_info.get("noise_mode", "diffuse") if model_info else "diffuse"
    rate = model_info.get("rate", None) if model_info else None

    for nv in noise_levels:
        if noise_mode == "diffuse":
            std = nv * np.sqrt(t)
        elif noise_mode == "decay":
            std = nv / np.sqrt(t)
        elif noise_mode == "power-decay":
            std = nv * (t ** (rate if rate is not None else 0.5))
        elif noise_mode == "exp-decay":
            std = nv * np.exp(t ** (rate if rate is not None else 1.0))
        else:
            continue

        ax5.plot(t, std, label=f"σ₀={nv:g}", alpha=0.7)

    ax5.set_title(f"STD schedule over time ({noise_mode})", fontsize=13)
    ax5.set_xlabel("Time (t)")
    ax5.set_ylabel("Standard deviation (σ)")
    ax5.set_yscale("log")
    ax5.legend(title="Initial σ₀", fontsize=8)
    ax5.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # Save everything including Fig5
    if save_base and model_info:
        save_dir = make_model_save_dir(save_base, model_info)
        figs = [fig1, fig2, fig3, fig4, fig5]
        save_all_figures(figs, save_dir, prefix=f"{model_info.get('model_name','')}_")
        print(f"✅ All figures saved to {save_dir}")
    else:
        print("⚠️ No save path or model_info provided; skipping save.")
    
# ===================== Histogram Plotting =====================
def plot_histograms_for_noise(nv, run_data, bins=60, xmax_percentile=99.5):
    """
    Plot histogram of hit vs FA scores for a single noise level.
    Automatically handles 'distance' vs 'likelihood' score types.
    """
    hits = run_data["hits"]
    fas  = run_data["fas"]
    score_type = run_data.get("score_type", "distance")

    pooled = np.concatenate([hits, fas]) if hits.size and fas.size else None
    if pooled is None or pooled.size == 0:
        print(f"[skip] No data for noise={nv}")
        return

    if score_type == "distance":
        xmax = float(np.percentile(pooled, xmax_percentile))
        rng = (0, xmax)
        xlabel = "Min distance"
        title  = f"Distance distributions | noise={nv:g}"
    else:  # likelihood (can be negative)
        xmin = float(np.percentile(pooled, 100 - xmax_percentile))
        xmax = float(np.percentile(pooled, xmax_percentile))
        rng = (xmin, xmax)
        xlabel = "Log likelihood"
        title  = f"Likelihood distributions | noise={nv:g}"

    fig, ax = plt.subplots(figsize=(6.5, 4.5))
    ax.hist(hits, bins=bins, range=rng, density=True, alpha=0.55,
            color='green', label=f'hits (N={len(hits)})')
    ax.hist(fas,  bins=bins, range=rng, density=True, alpha=0.55,
            color='red',   label=f'FAs (N={len(fas)})')

    ax.set_xlabel(xlabel)
    ax.set_ylabel("Density")
    ax.set_title(title)
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()

# ===================== Summary ROCs =====================
def roc_all_isis_across_noise(runs_by_noise):
    pooled_hits, pooled_fas = [], []
    score_type = None
    for run in runs_by_noise.values():
        pooled_hits.extend(run["hits"])
        pooled_fas.extend(run["fas"])
        # ensure consistent score_type
        if score_type is None:
            score_type = run.get("score_type", "distance")
    return roc_from_arrays(np.asarray(pooled_hits), np.asarray(pooled_fas),
                           score_type=score_type)


def roc_non1_isis_across_noise(runs_by_noise):
    pooled_hits, pooled_fas = [], []
    score_type = None
    for run in runs_by_noise.values():
        if "isi_hit_dists" in run:
            for isi, lst in run["isi_hit_dists"].items():
                if isi != 1:
                    pooled_hits.extend([d for (d, _t) in lst])
        else:
            pooled_hits.extend(run["hits"])
        pooled_fas.extend(run["fas"])
        if score_type is None:
            score_type = run.get("score_type", "distance")
    return roc_from_arrays(np.asarray(pooled_hits), np.asarray(pooled_fas),
                           score_type=score_type)


def plot_summary_rocs(runs_by_noise):
    """Plot both summary ROCs side by side."""
    fig, axes = plt.subplots(1, 2, figsize=(11, 5))

    # All ISIs
    res_all = roc_all_isis_across_noise(runs_by_noise)
    if res_all is not None:
        fpr, tpr, auc_val = res_all
        plot_roc(axes[0], fpr, tpr, label=f"All ISIs | AUC={auc_val:.3f}")
    axes[0].plot([0,1],[0,1],'k--',alpha=0.5)
    axes[0].set_title("Summary ROC (All ISIs)")
    axes[0].legend(loc="lower right")

    # Non-1 ISIs
    res_non1 = roc_non1_isis_across_noise(runs_by_noise)
    if res_non1 is not None:
        fpr, tpr, auc_val = res_non1
        plot_roc(axes[1], fpr, tpr, label=f"ISI≠1 | AUC={auc_val:.3f}")
    axes[1].plot([0,1],[0,1],'k--',alpha=0.5)
    axes[1].set_title("Summary ROC (Non-1 ISIs)")
    axes[1].legend(loc="lower right")

    for ax in axes:
        ax.set_xlabel("False Positive Rate")
        ax.set_ylabel("True Positive Rate")
        ax.grid(True, alpha=0.2)

    plt.tight_layout()
    plt.show()

In [ ]:
files = glob.glob("/mindhive/mcdermott/www/bjmedina/experiments/mem_exp_v02/results/*V15.csv")
exps_, seqs_, fnames_ = load_results_with_exclusion_2(
    "/mindhive/mcdermott/www/bjmedina/experiments/mem_exp_v02/results/V15",
    min_dprime=2, min_trials=120, skip_len60=True, verbose=False, return_skipped=False
)

base_path_ = "/mindhive/mcdermott/www/mturk_stimuli/bjmedina/mem_exp_v15/sequences/"

# Build experiment_list
experiment_list_ = []
for seq_ in seqs_:
    with open(base_path_ + seq_, 'r') as f:
        data = json.load(f)
    stim_paths = ["/mindhive/mcdermott/www/mturk_stimuli/bjmedina/mem_exp_v15/" + s 
                  for s in data['filenames_order']]

    experiment_list_.append(stim_paths)

In [ ]:
# ===================== Main Script =====================
# Example: load experiment_list and encode clean reps
# Replace with your own task selection and paths
which_task = "atexts-len120"
seqs_paths = {
    "ind-nature-len120": "mem_exp_ind-nature_2025", 
    "global-music-len120": "global-music-2025-n_80",
    "atexts-len120": "mem_exp_atexts_2025",
    "nhs-region-len120": "nhs-region-n_80"
}
base_path = "/mindhive/mcdermott/www/mturk_stimuli/bjmedina/{}/sequences/isi_16/len120/"
exps, seqs, fnames = load_results_with_exclusion_2(
    f"/mindhive/mcdermott/www/bjmedina/experiments/bolivia_2025/results/isi_16/{which_task}",
    min_dprime=2, min_trials=120, skip_len60=True, verbose=False, return_skipped=False
)
move_sequences_to_used(base_path.format(seqs_paths[which_task]), seqs)

# Build experiment_list
experiment_list = []
for seq in seqs:
    with open(base_path.format(seqs_paths[which_task]) + seq, 'r') as f:
        data = json.load(f)
    stim_paths = ["/mindhive/mcdermott/www/mturk_stimuli/bjmedina/" + seqs_paths[which_task] + "/" + s 
                  for s in data['filenames_order']]
    experiment_list.append(stim_paths)

In [ ]:
base_path = "/om/data/public/audioset/wavs/unbalanced_train_segments_downloads/"

# Load CSV
df = pd.read_csv("/orcd/data/jhm/001/om2/jmhicks/projects/TextureStreaming/stimuli/OVERLAPPED_0.5s_all_4s_sound_list_with_stationarity_score_no_speech_no_music_audioset_matlab_coch_rms0p02.csv")

# Filter only the rows that pass the stationarity threshold
df["pass_stationarity"] = df["stationarity_score"] <= -0.6

# Group by full filepath
grouped = df.groupby("filepath")["pass_stationarity"]

# Compute fraction of segments that pass per file
fraction_passing = grouped.mean()

# Keep files where more than 50% of segments pass
files_to_use = fraction_passing[fraction_passing > 0.5].index.tolist()
files_to_use = [base_path+f for f in files_to_use]

In [ ]:
# Encode clean reps
pc_texture_model = encoders.AudioTextureEncoderPCA(
    statistics_dict=statistics_set.statistics,
    pc_dims=PC_DIMS, model_params=model_params,
    sr=20000, rms_level=0.05, duration=2.0, device=DEVICE
)
zscore_projector = encoders.ZScoreSpace(pc_texture_model, device=DEVICE)
zscore_projector.fit(files_to_use[:100])

tmp = DistanceMemoryModel.DistanceMemoryModel(pc_texture_model, NV_DEFAULT, criterion=0.0, device=DEVICE)
all_files = sorted({fn for seq in experiment_list for fn in seq})
all_files_ = sorted({fn for seq in experiment_list_ for fn in seq})

tmp._fill_memory_bank(all_files)
with torch.no_grad():
    X0 = torch.stack([rep.detach().clone().view(-1) for rep in tmp.memory_bank], dim=0).to(DEVICE)

tmp = DistanceMemoryModel.DistanceMemoryModel(pc_texture_model, NV_DEFAULT, criterion=0.0, device=DEVICE)
tmp._fill_memory_bank(all_files_)
with torch.no_grad():
    X0_ = torch.stack([rep.detach().clone().view(-1) for rep in tmp.memory_bank], dim=0).to(DEVICE)
    
name_to_idx = {fn: i for i, fn in enumerate(all_files)}
name_to_idx_ = {fn: i for i, fn in enumerate(all_files_)}

save_base="/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/figures/model-results/"

In [ ]:
model_info = dict(
    model_name="DistanceMemoryModel",
    metric="euclidean",
    noise_mode="decay",
    encoder="texture_statistics",
    sigma0=0.0,
    run_id="prolific_batch"
)

NOISE_LEVELS = np.geomspace(0.1, 10, 10)

curves, runs = rocs_across_noise(
    NOISE_LEVELS,
    runner=run_experiment_scores,
    X0=X0_, name_to_idx=name_to_idx_, experiment_list=experiment_list_[:q0],
    metric="euclidean",
    noise_mode="decay"# or "loglikelihood"
)

save_dir = make_model_save_dir(save_base, model_info)

In [ ]:
# Then overlay all ROCs across noise levels:
plot_noise_overlays(curves, save_dir=save_dir)


plot_across_noise(NOISE_LEVELS, runs, bins=60, xmax_percentile=99.5, isis=(1,2,3,4,9,17,33,65),
                  model_info=model_info,
                  save_base=save_base)

In [ ]:
model_info = dict(
    model_name="DistanceMemoryModel",
    metric="euclidean",
    noise_mode="diffuse",
    encoder="texture_statistics",
    rate=0.0,
    run_id="prolific_batch"
)

NOISE_LEVELS = np.geomspace(0.01, 10, 10)[:5]

curves, runs = rocs_across_noise(
    NOISE_LEVELS,
    runner=run_experiment_scores,
    X0=X0_, name_to_idx=name_to_idx_, experiment_list=experiment_list_[:10],
    metric="euclidean",
    noise_mode="diffuse"# or "loglikelihood"
)

save_dir = make_model_save_dir(save_base, model_info)

# Then overlay all ROCs across noise levels:
plot_noise_overlays(curves, save_dir=save_dir)

plot_across_noise(NOISE_LEVELS, runs, bins=60, xmax_percentile=99.5, isis=(1,2,3,4,9,17,33,65),
                  model_info=model_info,
                  save_base=save_base)

In [ ]:
model_info = dict(
    model_name="DistanceMemoryModel",
    metric="euclidean",
    noise_mode="exp-decay",
    encoder="texture_statistics",
    rate=0.2,
    run_id="prolific_batch"
)

NOISE_LEVELS = np.geomspace(0.01, 10, 10)[:7]

curves, runs = rocs_across_noise(
    NOISE_LEVELS,
    rate=0.2,
    runner=run_experiment_scores,
    X0=X0_, name_to_idx=name_to_idx_, experiment_list=experiment_list_[:15],
    metric="euclidean",
    noise_mode="exp-decay"# or "loglikelihood"
)

save_dir = make_model_save_dir(save_base, model_info)

# Then overlay all ROCs across noise levels:
plot_noise_overlays(curves, save_dir=save_dir)

plot_across_noise(NOISE_LEVELS, runs, bins=60, xmax_percentile=99.5, isis=(1,2,3,4,9,17,33,65),
                  model_info=model_info,
                  save_base=save_base)

In [ ]:
model_info = dict(
    model_name="DistanceMemoryModel",
    metric="euclidean",
    noise_mode="exp-decay",
    encoder="texture_statistics",
    rate=0.1,
    run_id="prolific_batch"
)

NOISE_LEVELS = np.geomspace(0.01, 10, 10)[:7]

curves, runs = rocs_across_noise(
    NOISE_LEVELS,
    rate=0.2,
    runner=run_experiment_scores,
    X0=X0_, name_to_idx=name_to_idx_, experiment_list=experiment_list_[:15],
    metric="euclidean",
    noise_mode="exp-decay"# or "loglikelihood"
)

save_dir = make_model_save_dir(save_base, model_info)

# Then overlay all ROCs across noise levels:
plot_noise_overlays(curves, save_dir=save_dir)

plot_across_noise(NOISE_LEVELS, runs, bins=60, xmax_percentile=99.5, isis=(1,2,3,4,9,17,33,65),
                  model_info=model_info,
                  save_base=save_base)

In [ ]:
# NOISE_LEVELS = np.linspace(0.1, 1, 5)

# curves, runs = rocs_across_noise(
#     NOISE_LEVELS,
#     runner=run_experiment_scores,
#     X0=X0, name_to_idx=name_to_idx, experiment_list=experiment_list,
#     metric="mahalanobis",
#     noise_mode="decay"# or "loglikelihood"
# )

# plot_noise_overlays(curves)
# # for nv in NOISE_LEVELS:
# #     if nv in runs:
# #         plot_isi_and_half_for_noise(nv, runs[nv], isis=(1,17))
# #         plot_histograms_for_noise(nv, runs[nv], bins=60)

# plot_across_noise(NOISE_LEVELS, runs, bins=60, xmax_percentile=99.5, isis=(1,17))

In [ ]:
NOISE_LEVELS = np.linspace(0.1, 1, 4)

curves, runs = rocs_across_noise(
    NOISE_LEVELS,
    runner=run_experiment_scores,
    X0=X0, name_to_idx=name_to_idx, experiment_list=experiment_list,
    metric="loglikelihood",
    noise_mode="diffuse"#   # or "loglikelihood"
)

plot_noise_overlays(curves)
# for nv in NOISE_LEVELS:
#     if nv in runs:
#         plot_isi_and_half_for_noise(nv, runs[nv], isis=(1,17))
#         plot_histograms_for_noise(nv, runs[nv], bins=60)

plot_across_noise(NOISE_LEVELS, runs, bins=60, xmax_percentile=99.5, isis=(1,17))

In [ ]:
NOISE_LEVELS = np.linspace(0.1, 1, 4)

curves, runs = rocs_across_noise(
    NOISE_LEVELS,
    runner=run_experiment_scores,
    X0=X0, name_to_idx=name_to_idx, experiment_list=experiment_list,
    metric="loglikelihood",
    noise_mode="decay"#   # or "loglikelihood"
)

plot_noise_overlays(curves)
# for nv in NOISE_LEVELS:
#     if nv in runs:
#         plot_isi_and_half_for_noise(nv, runs[nv], isis=(1,17))
#         plot_histograms_for_noise(nv, runs[nv], bins=60)

plot_across_noise(NOISE_LEVELS, runs, bins=60, xmax_percentile=99.5, isis=(1,17))

In [ ]:
def run_experiment_noisy(
    sigma_enc,
    *,
    sigma0=0.2,          # fixed noise for drift
    epochs=5,
    metric="mahalanobis",
    X0=None, name_to_idx=None, experiment_list=None,
    debug=False, **kwargs
):
    """
    Runner with both drift noise (sigma0) and encoding noise (sigma_enc).
    
    Args:
        sigma_enc (float): encoding noise std (swept across noise_levels)
        sigma0 (float): drift noise std (fixed across runs)
        epochs (int): number of repeated runs
        metric (str): 'mahalanobis' or 'loglikelihood'
    """
    assert metric in {"mahalanobis", "loglikelihood"}, f"Unknown metric: {metric}"

    hit_scores, fa_scores = [], []
    isi_hit_dists = defaultdict(list)
    T_max = max((len(seq) for seq in experiment_list), default=0)
    fa_by_t = [[] for _ in range(T_max)]
    D = X0.shape[1]

    for _ in range(epochs):
        for seq in experiment_list:
            if len(seq) == 0:
                continue

            seq_idx = [name_to_idx[fn] for fn in seq]
            memory_bank, bank_set, last_seen = [], set(), {}

            for t, idx_incoming in enumerate(seq_idx, start=1):
                probe = X0[idx_incoming].view(1, -1)
                scores = []

                # compare probe to all items in memory
                for mem in memory_bank:
                    age = t - mem["t_inserted"]
                    if age <= 0:
                        continue
                    std = max(sigma0 * np.sqrt(age), 1e-6)
                    var = std**2

                    diff = probe - mem["mu"]
                    sqdist = torch.sum(diff**2).item()

                    if metric == "mahalanobis":
                        score = (sqdist**0.5) / std
                    else:  # loglikelihood
                        score = (
                            -0.5 * D * np.log(2 * np.pi)
                            - D * np.log(std)
                            - 0.5 * (sqdist / var)
                        )
                    scores.append(score)

                # classify as hit or FA
                if scores:
                    score_val = min(scores) if metric == "mahalanobis" else max(scores)
                    if idx_incoming in bank_set:
                        hit_scores.append(score_val)
                        isi = t - last_seen[idx_incoming]
                        isi_hit_dists[isi].append((score_val, t))
                    else:
                        fa_scores.append(score_val)
                        fa_by_t[t-1].append(score_val)

                # ---- ENCODING NOISE applied here ----
                noise = torch.randn_like(X0[idx_incoming]) * sigma_enc
                noisy_mu = X0[idx_incoming].detach().clone() + noise

                memory_bank.append({
                    "mu": noisy_mu,
                    "t_inserted": t
                })
                bank_set.add(idx_incoming)
                last_seen[idx_incoming] = t

            if debug and memory_bank:
                print("First mem:", memory_bank[0])
                print("Last mem:", memory_bank[-1])
                input()

    return {
        "hits": np.asarray(hit_scores, float),
        "fas": np.asarray(fa_scores, float),
        "isi_hit_dists": isi_hit_dists,
        "fa_by_t": fa_by_t,
        "T_max": T_max,
        "score_type": "likelihood" if metric == "loglikelihood" else "distance"
    }

# Fix sigma0 here
SIGMA0_FIXED = 0.5
NOISE_LEVELS = np.linspace(1, 2, 5)  # values for sigma_enc

curves, runs = rocs_across_noise(
    NOISE_LEVELS,
    runner=run_experiment_noisy,
    X0=X0, name_to_idx=name_to_idx, experiment_list=experiment_list,
    sigma0=SIGMA0_FIXED,    # fixed
    metric="loglikelihood", # or "mahalanobis"
    epochs=1
)